# Steam 할인 패턴 분석

- 이벤트 기준 132건
- 게임 기준 46개 함께 확인
- 반응률과 유지율은 Winsorize 적용
- 시즌 비교는 참고 비교로 해석


## Part 0 환경 설정

윈도우와 맥 모두 한글이 안 깨지게 폰트를 먼저 설정


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager

warnings.filterwarnings('ignore')


def root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / 'data').exists() and (candidate / 'figures').exists():
            return candidate
    return cwd


PROJECT_ROOT = root()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)


def kfont():
    candidate_paths = [
        PROJECT_ROOT / 'fonts' / 'NanumGothic.ttc',
        PROJECT_ROOT / 'fonts' / 'NanumGothic.ttf',
    ]
    for font_path in candidate_paths:
        if font_path.exists():
            try:
                font_manager.fontManager.addfont(str(font_path))
                return font_manager.FontProperties(fname=str(font_path)).get_name()
            except Exception:
                pass

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans KR']:
        if font_name in installed_fonts:
            return font_name
    return 'DejaVu Sans'


FONT_NAME = kfont()
plt.rcParams['font.family'] = FONT_NAME
plt.rcParams['font.sans-serif'] = [FONT_NAME, 'Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

FIG_W, FIG_H = 10, 6
DPI = 300

print('한글 폰트', FONT_NAME)
print('프로젝트 루트', PROJECT_ROOT)
print('저장 폴더', FIGURE_DIR)


## Part 1 데이터 준비

- 분석 테이블 불러오기
- Winsorize 적용
- 게임 단위 요약 테이블 생성

해석할 때는 이벤트 단위와 게임 단위를 같이 봄


In [ ]:
analysis_df = pd.read_csv(DATA_DIR / 'analysis_df.csv')
discount_history = pd.read_csv(DATA_DIR / 'discount_history.csv')
review_daily = pd.read_csv(DATA_DIR / 'review_daily.csv')

analysis_df['discount_start'] = pd.to_datetime(analysis_df['discount_start'])
analysis_df['discount_end'] = pd.to_datetime(analysis_df['discount_end'])
discount_history['discount_start'] = pd.to_datetime(discount_history['discount_start'])
discount_history['discount_end'] = pd.to_datetime(discount_history['discount_end'])
review_daily['date'] = pd.to_datetime(review_daily['date'])


def clip95(s, lower_pct=5, upper_pct=95):
    lo = np.percentile(s, lower_pct)
    hi = np.percentile(s, upper_pct)
    return s.clip(lo, hi), lo, hi


analysis_df['reaction_rate_raw'] = analysis_df['reaction_rate']
analysis_df['sustained_rate_raw'] = analysis_df['sustained_rate']
analysis_df['reaction_rate'], rr_lo, rr_hi = clip95(analysis_df['reaction_rate'])
analysis_df['sustained_rate'], sr_lo, sr_hi = clip95(analysis_df['sustained_rate'])

game_df = (
    analysis_df
    .groupby(['appid', 'name', 'genre_category'], as_index=False)
    .agg(
        n_events=('appid', 'count'),
        reaction_median=('reaction_rate', 'median'),
        sustained_median=('sustained_rate', 'median'),
        discount_mean=('discount_pct', 'mean'),
        has_seasonal=('is_seasonal_sale', 'max'),
    )
)

GENRE_ORDER = ['RPG', 'Adventure', 'Strategy/Simulation', 'Casual/Indie', 'Action']
GENRE_COLORS = {
    'RPG': '#4C72B0',
    'Adventure': '#DD8452',
    'Strategy/Simulation': '#55A868',
    'Casual/Indie': '#C44E52',
    'Action': '#8172B2',
}

print('이벤트 수', len(analysis_df))
print('게임 수', game_df['appid'].nunique())
print('Winsorize 경계', rr_lo, rr_hi, sr_lo, sr_hi)


## Part 2 장르별 반응

게임 단위 중앙값을 기준으로 장르별 반응을 비교

### 보는 법
- 막대 위 숫자는 반응률 중앙값
- `n=`은 그 장르에 포함된 게임 수
- 막대가 높을수록 그 장르의 반응이 더 큰 편
- 0 아래로 내려가면 할인 반응이 약하거나 음수인 경우가 많다는 뜻
- 이 차트는 게임 기준이라 반복 할인 게임의 영향이 조금 줄어든 상태


In [ ]:
genre_summary = (
    game_df.groupby('genre_category')['reaction_median']
    .agg(['median', 'count'])
    .reindex(GENRE_ORDER)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    genre_summary.index,
    genre_summary['median'],
    color=[GENRE_COLORS[g] for g in GENRE_ORDER],
    edgecolor='black',
    alpha=0.8,
)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('장르')
ax.set_ylabel('반응률 중앙값')
ax.set_title('장르별 반응')
ax.grid(axis='y', alpha=0.3)

for bar, genre in zip(bars, GENRE_ORDER):
    med = genre_summary.loc[genre, 'median']
    n = int(genre_summary.loc[genre, 'count'])
    ax.text(bar.get_x() + bar.get_width() / 2, med + 0.03, f'{med:.2f}\n n={n}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart1_genre.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart1_genre.png')


## Part 3 할인율과 반응

게임별 평균 할인율과 반응률 중앙값을 산점도로 비교

### 보는 법
- 점 하나가 게임 하나를 뜻함
- 오른쪽으로 갈수록 평균 할인율이 큼
- 위로 갈수록 반응률 중앙값이 큼
- 점들이 오른쪽 위 방향으로 모이면 할인율과 반응이 함께 커지는 경향이 있다는 뜻
- 상관계수는 전체 흐름을 참고하는 숫자일 뿐 원인 관계를 뜻하지 않음


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for genre in GENRE_ORDER:
    sub = game_df[game_df['genre_category'] == genre]
    ax.scatter(sub['discount_mean'], sub['reaction_median'], color=GENRE_COLORS[genre], label=genre, alpha=0.75, s=55)

rho, pval = spearmanr(game_df['discount_mean'], game_df['reaction_median'])
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('평균 할인율')
ax.set_ylabel('반응률 중앙값')
ax.set_title('할인율과 반응')
ax.legend(fontsize=9)
ax.text(0.02, 0.97, f'게임 기준\nSpearman ρ {rho:.2f}\np {pval:.3f}', transform=ax.transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart2_discount.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart2_discount.png')


## Part 4 반응과 유지

게임별 반응과 유지가 같이 움직이는지 확인

### 보는 법
- 점 하나가 게임 하나를 뜻함
- 오른쪽으로 갈수록 할인 중 반응이 큼
- 위로 갈수록 할인 후 유지가 큼
- 오른쪽 위는 반응도 크고 유지도 큰 경우
- 오른쪽 아래는 할인 중 반응은 컸지만 할인 후 유지가 약한 경우


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for genre in GENRE_ORDER:
    sub = game_df[game_df['genre_category'] == genre]
    ax.scatter(sub['reaction_median'], sub['sustained_median'], color=GENRE_COLORS[genre], label=genre, alpha=0.75, s=55)

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(game_df['reaction_median'].median(), color='gray', linestyle='--', linewidth=1)
ax.axhline(game_df['sustained_median'].median(), color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('반응률 중앙값')
ax.set_ylabel('유지율 중앙값')
ax.set_title('반응과 유지')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart3_keep.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart3_keep.png')


## Part 5 시즌 비교

시즌 이벤트가 있는 게임과 없는 게임을 참고 비교

### 보는 법
- 막대 위 숫자는 반응률 중앙값
- `n=`은 게임 수
- 시즌과 비시즌의 높이 차이는 참고 비교로만 봄
- 장르 구성과 게임 구성이 다를 수 있어서 시즌 효과라고 단정하면 안 됨


In [ ]:
season_summary = (
    game_df.groupby('has_seasonal')['reaction_median']
    .agg(['median', 'count'])
    .rename(index={False: '비시즌', True: '시즌'})
)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(season_summary.index, season_summary['median'], color=['#87CEEB', '#E9967A'], edgecolor='black', alpha=0.8)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('구분')
ax.set_ylabel('반응률 중앙값')
ax.set_title('시즌 참고 비교')
ax.grid(axis='y', alpha=0.3)

for bar, idx in zip(bars, season_summary.index):
    med = season_summary.loc[idx, 'median']
    n = int(season_summary.loc[idx, 'count'])
    ax.text(bar.get_x() + bar.get_width() / 2, med + 0.03, f'{med:.2f}\n n={n}', ha='center', va='bottom', fontsize=10)

ax.text(0.98, 0.03, '참고 비교\n장르 구성 차이 있음', transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart4_season.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart4_season.png')


## Part 6 사례 추이

설명용 사례 두 개를 골라 실제 리뷰 흐름을 확인

### 보는 법
- 위 그래프는 반응이 크게 나온 이벤트 사례
- 아래 그래프는 반복 할인이 많은 게임 사례
- 선이 위로 튈수록 그 시점의 일별 리뷰 수가 많다는 뜻
- 색칠된 구간은 할인 기간
- 사례는 이해를 돕기 위한 예시이지 전체를 대표한다고 단정하면 안 됨


In [ ]:
top_event = analysis_df.sort_values('reaction_rate', ascending=False).iloc[0]
repeat_game = analysis_df['appid'].value_counts().idxmax()
repeat_name = analysis_df.loc[analysis_df['appid'] == repeat_game, 'name'].iloc[0]

case1_start = top_event['discount_start'] - pd.Timedelta(days=30)
case1_end = top_event['discount_end'] + pd.Timedelta(days=14)
case1_reviews = review_daily[(review_daily['appid'] == top_event['appid']) & (review_daily['date'] >= case1_start) & (review_daily['date'] <= case1_end)].sort_values('date')
repeat_reviews = review_daily[review_daily['appid'] == repeat_game].sort_values('date')
repeat_events = discount_history[discount_history['appid'] == repeat_game].sort_values('discount_start')

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

axes[0].plot(case1_reviews['date'], case1_reviews['daily_reviews'], color='#355C7D', linewidth=1.8)
axes[0].axvspan(top_event['discount_start'], top_event['discount_end'], color='#F08A5D', alpha=0.25)
axes[0].set_title(f'사례 1  {top_event["name"]}')
axes[0].set_ylabel('일별 리뷰 수')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
axes[0].tick_params(axis='x', rotation=20)

axes[1].plot(repeat_reviews['date'], repeat_reviews['daily_reviews'], color='#2A9D8F', linewidth=1.5)
for _, ev in repeat_events.iterrows():
    axes[1].axvspan(ev['discount_start'], ev['discount_end'], color='#E76F51', alpha=0.18)
axes[1].set_title(f'사례 2  {repeat_name}')
axes[1].set_ylabel('일별 리뷰 수')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart5_cases.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart5_cases.png')


## 결론

- 장르에 따라 할인 반응은 다르게 보임
- 할인율이 높을수록 반응이 커질 때도 있음
- 반응과 유지는 같이 봐야 함
- 시즌 비교는 참고 비교로 해석
- 이 분석은 리뷰 반응 기준
